# Liquidity Risk

**Purpose:** Demonstrate the liquidity-risk helpers in `finstack_quant.portfolio` on a compact single-name example.

**Prerequisites:** Familiarity with volume, spread, VaR, and market-impact intuition.

**In this notebook:** We simulate prices and volumes, estimate several liquidity metrics, and then roll them into position-level and portfolio-level outputs.


## Concept

Liquidity is multi-dimensional. The same name can look liquid by spread but less liquid by days-to-liquidate or market impact. This notebook deliberately touches several lenses instead of treating liquidity as one scalar.


In [1]:
import numpy as np
import pandas as pd

from finstack_quant.analytics import Performance
from finstack_quant.core.math.special_functions import standard_normal_inv_cdf
from finstack_quant.portfolio import (
    almgren_chriss_impact,
    amihud_illiquidity,
    days_to_liquidate,
    kyle_lambda,
    liquidity_tier,
    lvar_bangia,
    roll_effective_spread,
)


def simulate_price_path(
    n_days: int = 100,
    initial_price: float = 100.0,
    annual_vol: float = 0.25,
    avg_daily_volume: float = 2_000_000.0,
    seed: int = 42,
):
    """Synthetic price/volume fixture with a bid-ask bounce baked into the path."""
    rng = np.random.default_rng(seed)
    daily_vol = annual_vol / np.sqrt(252.0)
    eff_returns = rng.normal(0.0, daily_vol, size=n_days)
    half_spread = 0.0005
    bounce = np.where(rng.random(n_days) < 0.5, half_spread, -half_spread)
    log_prices = np.cumsum(eff_returns) + bounce
    prices = initial_price * np.exp(log_prices)
    returns = np.diff(prices) / prices[:-1]
    volumes = rng.lognormal(mean=np.log(avg_daily_volume), sigma=0.35, size=n_days - 1)
    return prices, returns, volumes


## Spread, liquidation, VaR, and impact

The next cell follows the script in the same order an analyst would usually think about the problem: market-data diagnostics first, then liquidation speed, then VaR add-ons and execution cost, and finally a portfolio-level tiering view.

Two notes on where the numbers come from:

- **VaR is not hand-rolled.** The base (pre-liquidity) VaR that feeds `lvar_bangia()` comes from `analytics.Performance.parametric_var(0.99)`, and the volatility that feeds the impact model comes from `Performance.volatility(annualize=False)`. Both are computed in Rust, so the Gaussian quantile, the drift term, and the variance estimator all follow one convention. The `standard_normal_inv_cdf` line below is a *labelled cross-check* showing the zero-drift textbook form of the same quantity.
- **The Almgren-Chriss coefficients are illustrative, not canonical.** `almgren_chriss_impact()` takes the permanent (`gamma`) and temporary (`eta`) impact coefficients as inputs by design — calibrating them is a desk-specific exercise driven by your own execution data. The two one-line formulas below are a rough plausibility scaling so the example produces a sensible cost; do **not** treat them as a recommended calibration.


In [2]:
prices, returns, volumes = simulate_price_path()
print(f"Starting price     : {prices[0]:,.4f}")
print(f"Ending price       : {prices[-1]:,.4f}")
print(f"Mean daily volume  : {volumes.mean():,.0f}")

roll = roll_effective_spread(returns.tolist())
roll_for_impact = roll if roll is not None else 0.0010
amihud = amihud_illiquidity(returns.tolist(), volumes.tolist())
roll_display = f"{roll * 1e4:.2f}" if roll is not None else "unestimable"
print(f"\nRoll spread (bps)  : {roll_display}")
print(f"Amihud illiq.      : {amihud:.4e}")

position_value = 5_000_000.0
participation = 0.10
adv_notional = volumes.mean() * prices[-1]
dtl = days_to_liquidate(position_value, adv_notional, participation)
tier = liquidity_tier(dtl)
print(f"\nADV notional       : ${adv_notional:,.0f}")
print(f"Days to liquidate  : {dtl:.4f}")
print(f"Liquidity tier     : {tier}")

# Volatility and parametric VaR both come from the Rust analytics engine, so the
# variance estimator and the Gaussian quantile follow a single convention.
returns_df = pd.DataFrame(
    {"ASSET": returns},
    index=pd.bdate_range("2024-01-01", periods=len(returns)),
)
perf = Performance.from_returns(returns_df, freq="daily")
daily_vol = perf.volatility(annualize=False)[0]
var_99 = perf.parametric_var(0.99)[0] * position_value

lvar = lvar_bangia(
    var=var_99,
    spread_mean=0.0010,
    spread_vol=0.0003,
    confidence=0.99,
    position_value=position_value,
)
print(f"\nDaily volatility   : {daily_vol:.6f}")
print(f"Base VaR           : ${abs(lvar['var']):,.2f}")
print(f"Bangia LVaR        : ${abs(lvar['lvar']):,.2f}")
print(f"LVaR / VaR         : {lvar['lvar_ratio']:.4f}x")

# Labelled cross-check: the textbook zero-drift form of the same VaR. The 99%
# quantile comes from the Rust inverse normal CDF, not a memorised -2.326.
z_99 = standard_normal_inv_cdf(0.01)
print(f"  cross-check: z(1%) = {z_99:.4f}, zero-drift VaR = ${abs(z_99 * daily_vol * position_value):,.2f}")

shares = position_value / prices[-1]
# Illustrative impact-coefficient scaling only — see the note above.
gamma = roll_for_impact / (2.0 * volumes.mean())
eta = daily_vol * np.sqrt(prices[-1] / volumes.mean())
impact = almgren_chriss_impact(
    position_size=shares,
    avg_daily_volume=volumes.mean(),
    volatility=daily_vol,
    execution_horizon_days=5.0,
    permanent_impact_coef=gamma,
    temporary_impact_coef=eta,
)
kyle = kyle_lambda(volumes.tolist(), returns.tolist())
kyle_display = f"{kyle:.4e}" if kyle is not None else "unestimable"
print(f"\nAlmgren-Chriss bps : {impact['expected_cost_bps']:.2f}")
print(f"Kyle lambda        : {kyle_display}")

portfolio = [
    ("AAPL_5M", 5_000_000.0, 8e9),
    ("SMALL_CAP_5M", 5_000_000.0, 20_000_000.0),
    ("MICRO_CAP_2M", 2_000_000.0, 500_000.0),
    ("MEGA_CAP_50M", 50_000_000.0, 50e9),
]
print("\nPortfolio tiers")
for name, pv, adv in portfolio:
    days = days_to_liquidate(pv, adv, 0.10)
    print(f"  {name:<15} dtl={days:>8.3f}  tier={liquidity_tier(days)}")

Starting price     : 100.5313
Ending price       : 92.3424
Mean daily volume  : 2,100,312

Roll spread (bps)  : unestimable
Amihud illiq.      : 5.3697e-09

ADV notional       : $193,947,743
Days to liquidate  : 0.2578
Liquidity tier     : tier1

Daily volatility   : 0.012140
Base VaR           : $145,138.95
Bangia LVaR        : $149,383.71
LVaR / VaR         : 1.0292x
  cross-check: z(1%) = -2.3263, zero-drift VaR = $141,214.87

Almgren-Chriss bps : 83.84
Kyle lambda        : unestimable

Portfolio tiers
  AAPL_5M         dtl=   0.006  tier=tier1
  SMALL_CAP_5M    dtl=   2.500  tier=tier2
  MICRO_CAP_2M    dtl=  40.000  tier=tier4
  MEGA_CAP_50M    dtl=   0.010  tier=tier1


## Takeaways

- `roll_effective_spread()` and `amihud_illiquidity()` capture different aspects of trading frictions.
- `days_to_liquidate()` and `liquidity_tier()` translate raw market data into position-management language.
- `lvar_bangia()` and `almgren_chriss_impact()` are natural next steps once you want to fold liquidity into risk or execution decisions.
- The *inputs* to those two matter as much as the functions. Source the base VaR and volatility from `analytics.Performance` rather than re-deriving them, and treat the impact coefficients as a calibration you own.


In [3]:
{
    "roll_spread_bps": None if roll is None else round(roll * 1e4, 2),
    "days_to_liquidate": round(dtl, 4),
    "liquidity_tier": tier,
    "base_var": round(abs(lvar["var"]), 2),
    "bangia_lvar": round(abs(lvar["lvar"]), 2),
    "lvar_ratio": round(lvar["lvar_ratio"], 4),
    "impact_bps": round(impact["expected_cost_bps"], 2),
}

{'roll_spread_bps': None,
 'days_to_liquidate': 0.2578,
 'liquidity_tier': 'tier1',
 'base_var': 145138.95,
 'bangia_lvar': 149383.71,
 'lvar_ratio': 1.0292,
 'impact_bps': 83.84}